# Stage 00 — Universe and High-Recall Reproducibility

This notebook is a **read-only evidence walkthrough** for the governed Stage 00 chain: acquisition lineage → issuer exclusions → Item 1 packets → high-recall-screen readiness. It is not a second implementation of the pipeline. Production logic remains in `src/` and `pipelines/`; immutable artifacts remain under `data/runs/`.

Default mode is `verify`: no SEC request, model call, credential resolution, run-directory creation, or artifact write occurs.

## What this notebook can establish

Given the canonical artifacts present locally, it re-hashes the v5 packet manifest and output JSONL files, checks the cohort ledger, verifies the canary selection binding, and summarizes prior non-authoritative canary receipts. It can therefore reproduce the *verification result* deterministically.

It cannot promise byte-identical results from a future live model call: provider behavior is external. A future live run must still use the governed CLI, a fresh run id, and a separately minted authorization.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
from pathlib import Path

def find_repo_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Open this notebook from the repository or one of its subdirectories.')

REPO_ROOT = find_repo_root()
print(f'Repository: {REPO_ROOT}')

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def jsonl_row_count(path: Path) -> int:
    with path.open('rb') as handle:
        return sum(1 for _ in handle)


## 1. Repository identity and safe mode

A dirty worktree is reported rather than repaired. This notebook never stages, commits, pushes, or mutates repository files.

In [ ]:
def git(*args: str) -> str:
    return subprocess.check_output(['git', *args], cwd=REPO_ROOT, text=True).strip()

head = git('rev-parse', 'HEAD')
branch = git('branch', '--show-current')
status = git('status', '--short', '--untracked-files=all')
print(f'branch: {branch}')
print(f'HEAD:   {head}')
print('worktree: clean' if not status else f'worktree has changes:\n{status}')

EXECUTION_MODE = 'verify'  # This notebook intentionally supports only read-only verification.
assert EXECUTION_MODE == 'verify'


## 2. Canonical v5 packet corpus

The v5 manifest is the current Stage 00 screening corpus authority. It binds the acquisition aggregate, shell and asset-backed determination inputs, Item 1 locator, output hashes, and cohort accounting.

In [ ]:
PACKET_RUN = REPO_ROOT / 'data/runs/baseline-packets/baseline-packets-domestic-text-lineage-v5-20260819'
PACKET_MANIFEST_PATH = PACKET_RUN / 'baseline_packet_manifest.json'
EXPECTED_PACKET_MANIFEST_SHA256 = '516b7020c657a7b656880444e0f98479c1aa46dca80bda9a1beafd846d7d88d8'

assert PACKET_MANIFEST_PATH.is_file(), f'Missing canonical manifest: {PACKET_MANIFEST_PATH}'
assert sha256_file(PACKET_MANIFEST_PATH) == EXPECTED_PACKET_MANIFEST_SHA256
PACKET_MANIFEST = load_json(PACKET_MANIFEST_PATH)
COUNTS = PACKET_MANIFEST['counts']

print('manifest SHA-256:', sha256_file(PACKET_MANIFEST_PATH))
print('packet contract:', PACKET_MANIFEST['schema_versions']['baseline_packet_manifest_v5'])
print('Item 1 locator:', PACKET_MANIFEST['item_one_locator'])
print('aggregate SHA:', PACKET_MANIFEST['aggregate_manifest_sha256'])
print('shell SHA:', PACKET_MANIFEST['shell_determination_manifest_sha256'])
print('asset-backed SHA:', PACKET_MANIFEST['asset_backed_determination_manifest_sha256'])


In [ ]:
expected_ledger = {
    'planned_rows': 8718,
    'firms_excluded': 1146,
    'retained_rows': 7572,
    'packets_built': 7042,
    'packet_failures': 530,
    'shell_only_true': 795,
    'asset_backed_only_true': 351,
    'both_true': 0,
}
for key, expected in expected_ledger.items():
    assert COUNTS[key] == expected, (key, COUNTS[key], expected)
assert COUNTS['planned_rows'] == COUNTS['firms_excluded'] + COUNTS['retained_rows']
assert COUNTS['retained_rows'] == COUNTS['packets_built'] + COUNTS['packet_failures']
assert COUNTS['firms_excluded'] == (COUNTS['shell_only_true']
                                    + COUNTS['asset_backed_only_true']
                                    + COUNTS['both_true'])

for key in ('planned_rows', 'firms_excluded', 'retained_rows', 'packets_built', 'packet_failures'):
    print(f'{key:>18}: {COUNTS[key]:,}')
print('failure reasons:', COUNTS['failures_by_reason'])
print('passages total:', f"{COUNTS['passages_total']:,}")


## 3. Re-hash the two v5 outputs

This reads the packet and failure JSONL files but does not alter them. On a normal local disk the packet JSONL check is the slowest cell because it is roughly 500 MB.

In [ ]:
for filename, expected_sha in PACKET_MANIFEST['output_hashes'].items():
    path = PACKET_RUN / filename
    assert path.is_file(), f'Missing output: {path}'
    observed_sha = sha256_file(path)
    assert observed_sha == expected_sha, (filename, observed_sha, expected_sha)
    rows = jsonl_row_count(path)
    print(f'{filename}: {rows:,} rows; SHA-256 verified')

assert jsonl_row_count(PACKET_RUN / 'universe_baseline_packets.jsonl') == COUNTS['packets_built']
assert jsonl_row_count(PACKET_RUN / 'baseline_packet_failures.jsonl') == COUNTS['packet_failures']


## 4. High-recall canary selection

The selection is a deterministic, packet-native, 100-row canary. It is not a software-universe release and does not authorize a live call by itself.

In [ ]:
SELECTION_PATH = REPO_ROOT / 'data/runs/universe-screen-selections/universe-screen-canary-selection-v1-20260819/universe_screen_selection.json'
EXPECTED_SELECTION_SHA256 = '26bd88052a4efd9c8b5580411c7fd9a2054d27314bae277799a0a7a0a4b0d570'
assert sha256_file(SELECTION_PATH) == EXPECTED_SELECTION_SHA256
SELECTION = load_json(SELECTION_PATH)
assert SELECTION['selection_kind'] == 'canary_100'
assert len(SELECTION['rows']) == 100
assert SELECTION['packet_manifest_sha256'] == EXPECTED_PACKET_MANIFEST_SHA256
assert len({(row['cik'], row['accession'], row['packet_sha256']) for row in SELECTION['rows']}) == 100
print('selection SHA-256:', sha256_file(SELECTION_PATH))
print('algorithm:', SELECTION['sampling']['algorithm'])
print('seed:', SELECTION['sampling']['seed'])
print('strata:', len(SELECTION['sampling']['strata']))


## 5. Prior authoritative-canary receipts

Receipt-bearing directories are evidence of calibration only. They are non-authoritative by construction: they lack an authoritative manifest and must never become a classifier input.

In [ ]:
SCREEN_RUNS_ROOT = REPO_ROOT / 'data/runs/universe-screens'
for directory in sorted(path for path in SCREEN_RUNS_ROOT.iterdir() if path.is_dir()):
    receipt_path = directory / 'universe_screen_failure_receipt.json'
    authoritative_manifest = directory / 'universe_screen_manifest.json'
    if receipt_path.is_file():
        receipt = load_json(receipt_path)
        assert not authoritative_manifest.exists(), directory
        print(f"{directory.name}: receipt={receipt['reason_code']}, row={receipt['stopping_row_index']}, authoritative=False")
    else:
        print(f'{directory.name}: no authoritative manifest/receipt summary available')


## 6. What comes next

ADR-112’s diagnostic route is the next governed measurement: a separate diagnostic authorization will bind this v5 corpus, this selection, and prompt v3. Its result can identify the distribution of strict-validator outcomes across 100 rows. It cannot itself become `SCREEN_v1`.

The full authoritative high-recall run remains a later, separately authorized operation after diagnostic review and acceptance gates.

In [ ]:
DIAGNOSTIC_COMMAND_TEMPLATE = [
    'python', 'pipelines/00_build_company_universe.py',
    '--mode', 'screen-universe-lineage-diagnostic',
    '--packet-manifest', str(PACKET_MANIFEST_PATH),
    '--selection-artifact', str(SELECTION_PATH),
    '--governance-root', '<fresh diagnostic governance root>',
    '--screen-authorization', 'screen_diagnostic_authorization.json',
    '--screen-authorization-sha256', '<fresh authorization sha256>',
    '--logical-request-cap', '100',
    '--provider-attempt-cap', '300',
    '--run-id', '<fresh unused diagnostic run id>',
]
print('Command template only — intentionally not executed:')
print(' '.join(DIAGNOSTIC_COMMAND_TEMPLATE))


## Reproducibility boundary

A researcher with this checkout and the same immutable artifacts can reproduce every check in this notebook. A future model run is reproducible as a governed experiment — exact inputs, route, prompt, authorization, captures, and outputs are recorded — but not assumed byte-identical merely because an external model provider is called again.